# Walmart Store Sales — Prophet v6: historical blend-weight tuning

v4 is the Prophet champion (`1367.45` WMAE). v5 showed that external covariates weakened this per-series model, so v6 returns to v4's event-aware calendar and tunes only the blend weight safely.

A 26-week historical calibration period chooses alpha without using the final 39-week validation target:

```text
prediction = (1 - alpha) × SeasonalNaive52 + alpha × raw Prophet
```


In [ ]:
%pip install -q "prophet>=1.1,<2" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
from __future__ import annotations
import json, logging, time, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from prophet import Prophet
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(exc)


In [ ]:
SEED=42
np.random.seed(SEED)
HOLIDAY_WINDOWS={"super_bowl":(0,0),"labor_day":(0,0),"thanksgiving":(-7,0),"christmas":(-7,0)}
ALPHA_GRID=np.round(np.arange(0.0,1.01,0.05),2).tolist()
CONFIG={
 "validation_weeks":39,"calibration_weeks":26,"holiday_weight":5.0,"top_n_series":None,"min_history_points":52,
 "seasonal_lag_weeks":52,"alpha_grid":ALPHA_GRID,"growth":"linear","yearly_seasonality":True,
 "weekly_seasonality":False,"daily_seasonality":False,"seasonality_mode":"additive",
 "changepoint_prior_scale":0.05,"seasonality_prior_scale":10.0,"holidays_prior_scale":10.0,
 "prediction_clip_min":0.0,"prediction_clip_max":300000.0,
 "wandb_project":"Walmart-Recruiting---Store-Sales-Forecasting","wandb_entity":"kende23-n-a",
 "wandb_group":"prophet-experiments","run_name":"prophet_v6_historical_alpha_tuning_all_series",
 "artifact_name":"prophet-v6-historical-alpha-tuning-all-series-validation"}
DATA_DIR=Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR=Path("/content/artifacts/prophet_v6_historical_alpha_tuning"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CONFIG


## Load, split and event-aware calendar


In [ ]:
train_raw=pd.read_csv(DATA_DIR/"train.csv",parse_dates=["Date"]).sort_values(["Store","Dept","Date"])
test_raw=pd.read_csv(DATA_DIR/"test.csv",parse_dates=["Date"])
all_dates=pd.Index(sorted(train_raw.Date.unique()),name="Date")
test_dates=pd.Index(sorted(test_raw.Date.unique()),name="Date")
final_dates=all_dates[-CONFIG["validation_weeks"]:]
final_fit_dates=all_dates[:-CONFIG["validation_weeks"]]
cal_dates=final_fit_dates[-CONFIG["calibration_weeks"]:]
cal_fit_dates=final_fit_dates[:-CONFIG["calibration_weeks"]]
if len(cal_fit_dates)<CONFIG["min_history_points"]: raise ValueError("Not enough calibration history")
series_sales=train_raw.groupby(["Store","Dept"],as_index=False).Weekly_Sales.sum().sort_values("Weekly_Sales",ascending=False)
top_series=series_sales[["Store","Dept"]] if CONFIG["top_n_series"] is None else series_sales.head(CONFIG["top_n_series"])[["Store","Dept"]]
selected=train_raw.merge(top_series,on=["Store","Dept"],how="inner")
panel=selected.pivot_table(index=["Store","Dept"],columns="Date",values="Weekly_Sales",aggfunc="sum").reindex(index=pd.MultiIndex.from_frame(top_series),columns=all_dates).fillna(0.0)
holiday_by_date=train_raw[["Date","IsHoliday"]].drop_duplicates("Date").set_index("Date").reindex(all_dates).IsHoliday.fillna(False).astype(bool)
def event_name(d):
 m=pd.Timestamp(d).month
 return {2:"super_bowl",9:"labor_day",11:"thanksgiving",12:"christmas"}.get(m,"other_walmart_holiday")
cal=pd.concat([train_raw[["Date","IsHoliday"]],test_raw[["Date","IsHoliday"]]]).drop_duplicates("Date")
holidays=cal[cal.IsHoliday].copy(); holidays["ds"]=pd.to_datetime(holidays.Date); holidays["holiday"]=holidays.ds.map(event_name)
holidays[["lower_window","upper_window"]]=holidays.holiday.map(lambda x:HOLIDAY_WINDOWS.get(x,(0,0))).apply(pd.Series)
holidays=holidays[["holiday","ds","lower_window","upper_window"]]
print({"series":len(panel),"calibration":(str(cal_dates.min().date()),str(cal_dates.max().date())),"final_validation":(str(final_dates.min().date()),str(final_dates.max().date()))})


## Prophet helpers and historical calibration


In [ ]:
def wmae(y,p,h):
 w=np.where(np.asarray(h,dtype=bool),CONFIG["holiday_weight"],1.0)
 return float(np.sum(w*np.abs(np.asarray(y)-np.asarray(p)))/np.sum(w))
def seasonal(series, dates):
 pos=all_dates.get_indexer(dates); lag=CONFIG["seasonal_lag_weeks"]
 if (pos<lag).any(): raise ValueError("seasonal lag unavailable")
 return series.loc[all_dates[pos-lag]].to_numpy(float)
def raw_prophet(series, train_dates, forecast_dates):
 hist=pd.DataFrame({"ds":train_dates,"y":series.loc[train_dates].to_numpy(float)})
 hist.y=hist.y.clip(lower=0)
 base=seasonal(series,forecast_dates)
 if (hist.y!=0).sum()<2: return base,"fallback_insufficient_history"
 try:
  model=Prophet(growth=CONFIG["growth"],yearly_seasonality=CONFIG["yearly_seasonality"],weekly_seasonality=False,daily_seasonality=False,seasonality_mode=CONFIG["seasonality_mode"],changepoint_prior_scale=CONFIG["changepoint_prior_scale"],seasonality_prior_scale=CONFIG["seasonality_prior_scale"],holidays_prior_scale=CONFIG["holidays_prior_scale"],holidays=holidays)
  model.fit(hist); p=model.predict(pd.DataFrame({"ds":forecast_dates})).yhat.to_numpy(float)
  return np.clip(np.nan_to_num(p,nan=0,posinf=CONFIG["prediction_clip_max"],neginf=0),0,CONFIG["prediction_clip_max"]),"fit"
 except Exception: return base,"fallback_fit_error"
start=time.time(); cal_rows=[]; cal_status=[]
for n,((store,dept),series) in enumerate(panel.iterrows(),1):
 raw,status=raw_prophet(series,cal_fit_dates,cal_dates); base=seasonal(series,cal_dates); actual=series.loc[cal_dates].to_numpy(float)
 cal_rows.append(pd.DataFrame({"Store":store,"Dept":dept,"Date":cal_dates,"Actual":actual,"Seasonal":base,"RawProphet":raw,"IsHoliday":holiday_by_date.loc[cal_dates].to_numpy()})); cal_status.append(status)
 if n%250==0 or n==len(panel): print({"calibration_finished":n,"elapsed_min":round((time.time()-start)/60,2)})
cal_df=pd.concat(cal_rows,ignore_index=True)
alpha_rows=[]
for alpha in CONFIG["alpha_grid"]:
 pred=(1-alpha)*cal_df.Seasonal.to_numpy()+alpha*cal_df.RawProphet.to_numpy()
 alpha_rows.append({"alpha":float(alpha),"calibration_wmae":wmae(cal_df.Actual,pred,cal_df.IsHoliday)})
alpha_df=pd.DataFrame(alpha_rows).sort_values("calibration_wmae").reset_index(drop=True)
best_alpha=float(alpha_df.iloc[0].alpha)
print({"best_alpha":best_alpha,"calibration_wmae":float(alpha_df.iloc[0].calibration_wmae)})
display(alpha_df)


## Final untouched validation with calibrated alpha


In [ ]:
final_rows=[]; final_status=[]; final_start=time.time()
for n,((store,dept),series) in enumerate(panel.iterrows(),1):
 raw,status=raw_prophet(series,final_fit_dates,final_dates); base=seasonal(series,final_dates); actual=series.loc[final_dates].to_numpy(float)
 pred=np.clip((1-best_alpha)*base+best_alpha*raw,0,CONFIG["prediction_clip_max"])
 final_rows.append(pd.DataFrame({"Store":store,"Dept":dept,"Date":final_dates,"Weekly_Sales":actual,"SeasonalNaive52":base,"RawProphetPrediction":raw,"Prediction":pred,"IsHoliday":holiday_by_date.loc[final_dates].to_numpy(),"ModelStatus":status}))
 final_status.append(status)
 if n%250==0 or n==len(panel): print({"final_finished":n,"elapsed_min":round((time.time()-final_start)/60,2)})
val_df=pd.concat(final_rows,ignore_index=True); val_df["AbsError"]=np.abs(val_df.Weekly_Sales-val_df.Prediction)
blend_wmae=wmae(val_df.Weekly_Sales,val_df.Prediction,val_df.IsHoliday)
raw_wmae=wmae(val_df.Weekly_Sales,val_df.RawProphetPrediction,val_df.IsHoliday)
naive_wmae=wmae(val_df.Weekly_Sales,val_df.SeasonalNaive52,val_df.IsHoliday)
metrics={"validation/wmae":blend_wmae,"validation/raw_prophet_wmae":raw_wmae,"validation/seasonal_naive_wmae":naive_wmae,"validation/improvement_vs_v4_pct":100*(1367.44697173274-blend_wmae)/1367.44697173274,"tuning/best_alpha":best_alpha,"tuning/calibration_wmae":float(alpha_df.iloc[0].calibration_wmae),"fit/series_total":len(panel),"fit/series_fit_ok":int(sum(s=="fit" for s in final_status)),"fit/series_fallback":int(sum(s!="fit" for s in final_status)),"fit/elapsed_minutes":(time.time()-start)/60}
print(metrics)


## W&B logging


In [ ]:
run=wandb.init(entity=CONFIG["wandb_entity"],project=CONFIG["wandb_project"],group=CONFIG["wandb_group"],name=CONFIG["run_name"],job_type="hparam_tuning",tags=["prophet","v6","historical-backtest","alpha-tuning","event-aware","all-series"],config=CONFIG,save_code=True)
val_path=OUTPUT_DIR/"prophet_v6_validation_predictions.csv"; alpha_path=OUTPUT_DIR/"prophet_v6_alpha_calibration.csv"; metrics_path=OUTPUT_DIR/"prophet_v6_metrics.json"
val_df.to_csv(val_path,index=False); alpha_df.to_csv(alpha_path,index=False); metrics_path.write_text(json.dumps(metrics,indent=2))
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(alpha_df.sort_values("alpha").alpha,alpha_df.sort_values("alpha").calibration_wmae,marker="o"); ax.axvline(best_alpha,color="red"); ax.set_title("Historical calibration: alpha vs WMAE"); ax.set_xlabel("Prophet blend weight"); ax.set_ylabel("WMAE"); plt.tight_layout(); plot_path=OUTPUT_DIR/"prophet_v6_alpha_curve.png"; fig.savefig(plot_path,dpi=160); plt.show()
artifact=wandb.Artifact(CONFIG["artifact_name"],type="model-evaluation",metadata={**CONFIG,**metrics})
for p in [val_path,alpha_path,metrics_path,plot_path]: artifact.add_file(str(p))
run.log_artifact(artifact,aliases=["v6","validation","latest"])
run.log({**metrics,"tuning/alpha_table":wandb.Table(dataframe=alpha_df),"validation/predictions":wandb.Table(dataframe=val_df.sample(min(20000,len(val_df)),random_state=SEED)),"tuning/alpha_curve":wandb.Image(str(plot_path))})
run.summary.update(metrics); run.finish()
